In [1]:
import logging
import os

os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
from napistu.genomics.scverse_loading import DatasetsConfig
import numpy as np

from napistu_torch.load.constants import FM_DEFS, SCGPT_DEFS
from napistu_torch.load.foundation_models import FoundationModel
from napistu_torch.load.foundation_model_etl import process_scgpt

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


In [2]:
# Configuration
DATA_DIR = "data"
OUTPUT_DIR = "output"
# Raw config dictionary
DATASETS_CONFIG = {
    "efthymiou2025": {
        "uri": "https://cellxgene.cziscience.com/collections/6b701826-37bb-4356-9792-ff41fc4c3161",
        "path": os.path.expanduser("~/Desktop/DATA/genomics/efthymiou.h5ad")
    }
}

MODEL_PATH = os.path.join(DATA_DIR, "scGPT_bc")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validated config
datasets_config = DatasetsConfig(DATASETS_CONFIG)


In [ ]:
process_scgpt(MODEL_PATH, OUTPUT_DIR, datasets_config=datasets_config)


In [18]:
x = foundation_model.dataset_gene_embeddings["efthymiou2025"].get('scGPT/efthymiou2025/adipocyte (0)')
x.category

'adipocyte (0)'

In [ ]:
# Load FoundationModel
foundation_model = FoundationModel.load(OUTPUT_DIR, SCGPT_DEFS.MODEL_NAME)
embeddings = foundation_model.weights.static_gene_embeddings

GENES_OF_INTEREST = embeddings.gene_annotations[FM_DEFS.VOCAB_NAME].sample(20000).tolist()
GENE_MASK = [x in GENES_OF_INTEREST for x in embeddings.ordered_gene_ids]

# Compute attention on demand using FoundationModelWeights method
# This handles multi-head attention properly
layer_11_attn = foundation_model.weights.compute_attention_from_weights(
    layer_idx=11,
    n_heads=foundation_model.n_heads,
    gene_mask=GENE_MASK
)

foundation_model.dataset_gene_embeddings
foundation_model.weights.static_gene_embeddings